# e2 · Macro value in Elastic Net (block B2)

Do the 11 macro variables, including VIX, improve forecasts over past log RV alone?
Compare `en_full_macro` (`a`) with `en_no_macro` (`b`), using `research_config.BLOCKS["B2"]`.

**Tests:**
Log-RV MSE uses Clark-West, with DM for the raw loss difference. QLIKE uses DM.
Pooled differentials are averaged across ETFs each month and tested with Newey-West
(Andrews bandwidth). P-values are one-sided, H1: `a` beats `b`.

Clark-West is approximate for tuned Elastic Net and can be significant despite
worse raw MSE. DM p-values also require caution for nested comparisons.

Output in `forecasting/results/`: `e2_losses.csv`, `e2_evaluation.csv`.
`gain_t > 0` favours `a`; `loss_ratio` is loss(a) / loss(b).
`pct_change = (loss_ratio - 1) * 100`; negative means lower raw loss for `a`.


In [22]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import sys
from importlib import reload
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

from utils.paths import FORECAST_DIR, FORECAST_RESULTS_DIR as RESULTS_DIR
import research_config

# Notebook kernels cache earlier comparison definitions between runs.
reload(research_config)
from research_config import BLOCKS, TICKERS

RESULTS_DIR.mkdir(exist_ok=True)

block = BLOCKS["B2"]
models = block["models"]
pairs = block["pairs"]
df = pd.read_csv(FORECAST_DIR / "en_forecasts.csv", parse_dates=["date", "target_date"])
df = df[df["model"].isin(models) & df["etf"].isin(TICKERS)].copy()
missing = set(models) - set(df["model"])
if missing:
    raise ValueError(f"Missing required models: {sorted(missing)}")
if df.duplicated(["model", "etf", "date"]).any():
    raise ValueError("Duplicate model/ETF/forecast-month rows.")
if df[["date", "target_date"]].isna().any().any():
    raise ValueError("Forecast and target dates must be present.")
print(f"{len(pairs)} B2 pairs runnable")

1 B2 pairs runnable


In [23]:
def andrews_bandwidth(x):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    lag = max(0, min(lag, n - 1))
    s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where both variants have matching target dates and realized values.


In [24]:
d = df[df.groupby(["etf", "date"])["model"].transform("nunique") == len(models)].copy()
d = d.sort_values(["date", "etf", "model"]).reset_index(drop=True)
if d.empty:
    raise ValueError("No common forecast sample across the two models.")
values = ["actual_logRV", "actual_RV", "fc_log", "fc_level"]
if not np.isfinite(d[values].to_numpy()).all():
    raise ValueError("The common sample contains non-finite forecasts or outcomes.")
if (d[["actual_RV", "fc_level"]] <= 0).any().any():
    raise ValueError("QLIKE requires positive realized and forecast variances.")
keys = ["etf", "date"]
if (d.groupby(keys)["target_date"].nunique() != 1).any():
    raise ValueError("Target months differ across models.")
for col in ["actual_logRV", "actual_RV"]:
    wide = d.pivot(index=keys, columns="model", values=col)
    if not np.allclose(wide.to_numpy(), wide[[models[0]]].to_numpy(), rtol=1e-10, atol=1e-12):
        raise ValueError(f"Realized outcomes differ across models: {col}")
# Keep the pooled monthly average equally weighted across the five ETFs.
if (d.groupby("date")["etf"].nunique() != len(TICKERS)).any():
    raise ValueError("The common sample must contain all five ETFs each month.")

d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"Forecast origins: {d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}")
print(f"Target months: {d['target_date'].min():%Y-%m} to {d['target_date'].max():%Y-%m}")
print(f"{d['date'].nunique()} months × {len(TICKERS)} ETFs × {len(models)} models = {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e2_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)


Forecast origins: 2008-02 to 2025-11
Target months: 2008-03 to 2025-12
214 months × 5 ETFs × 2 models = 2140 rows


,QLIKE,MSE_log,MSE_level
model,,,
en_full_macro,0.3247,0.4902,0.0011
en_no_macro,0.3266,0.5228,0.0001


## Tests


In [25]:
loss_idx = losses.set_index(["model", "ETF"])
rows = []
cw_components = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"]).sort_index()
    B = d[d["model"] == b].set_index(["date", "etf"]).sort_index()
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    mse_diff = A["MSE_log"] - B["MSE_log"]
    comparisons = [("QLIKE", "DM", dm_diff), ("MSE_log", "DM", mse_diff)]
    # Clark-West adjusted differential; positive favours a.
    adjustment = (B["fc_log"] - A["fc_log"]) ** 2
    cw = -mse_diff + adjustment
    comparisons.append(("MSE_log", "Clark-West", cw))
    # Reuse these components in the diagnostic table.
    cw_components.append({
        "pair": pair_id,
        "mean_raw_gain": (-mse_diff).groupby("date").mean().mean(),
        "mean_adjustment_term": adjustment.groupby("date").mean().mean(),
        "mean_CW_differential": cw.groupby("date").mean().mean(),
    })

    for etf in ["pooled"] + TICKERS:
        for loss, test, diff in comparisons:
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            series = series.sort_index()
            if len(series) < 2:
                raise ValueError(f"Insufficient observations for {pair_id}, {etf}.")
            t = -hac_t(series) if test == "DM" else hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": float(norm.sf(t)),
            })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100  # negative: a has the lower raw loss
tests.to_csv(RESULTS_DIR / "e2_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value,pct_change
0,B2.1,en_full_macro,en_no_macro,QLIKE,DM,0.9941,0.0853,0.4660,-0.5909
1,B2.1,en_full_macro,en_no_macro,MSE_log,DM,0.9376,0.9533,0.1702,-6.2358
2,B2.1,en_full_macro,en_no_macro,MSE_log,Clark-West,0.9376,5.9284,0.0000,-6.2358


## By sector tests


In [26]:
from IPython.display import display

SECTORS = {
    "XLB": "Materials", "XLE": "Energy", "XLF": "Financials",
    "XLI": "Industrials", "XLY": "Cons. Disc.",
}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss, test):
    t = tests[tests["loss"].eq(loss) & tests["test"].eq(test)].copy()
    t["cell"] = [f"{ratio:.3f} ({fmt_p(p)})" for ratio, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell").reindex(columns=TICKERS + ["pooled"])
    table.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "DM"), ("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss, test))

MSE_log: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs en_no_macro,0.910 (0.082),1.038 (0.696),0.923 (0.105),0.861 (0.055),0.974 (0.371),0.938 (0.170)


MSE_log: ratio (p-value), Clark-West


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs en_no_macro,0.910 (<0.001),1.038 (0.271),0.923 (<0.001),0.861 (<0.001),0.974 (<0.001),0.938 (<0.001)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs en_no_macro,0.944 (0.228),1.206 (0.873),0.972 (0.324),0.886 (0.019),0.986 (0.394),0.994 (0.466)


## By sector percentage changes

Percentage change in mean loss for `en_full_macro` relative to `en_no_macro`:
`100 × (loss_ratio - 1)`. Negative values mean lower loss with macro variables;
positive values mean higher loss. Log MSE and QLIKE use the same common sample
as the tests above. The pooled row includes all five sectors.


In [27]:
for pair_id, a, b, _ in pairs:
    sector_pct_change = (
        tests.loc[tests["pair"].eq(pair_id) & tests["test"].eq("DM")]
        .pivot(index="ETF", columns="loss", values="pct_change")
        .reindex(index=TICKERS + ["pooled"], columns=["MSE_log", "QLIKE"])
        .rename(columns={"MSE_log": "Log MSE"})
    )
    sector_pct_change.index = pd.Index(
        [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"],
        name="Sector ETF",
    )
    sector_pct_change.columns.name = None
    print(f"{pair_id}: {a} relative to {b}")
    display(sector_pct_change.map(lambda value: f"{value:+.2f}%"))


B2.1: en_full_macro relative to en_no_macro


,Log MSE,QLIKE
Sector ETF,,
XLB Materials,-9.03%,-5.63%
XLE Energy,+3.76%,+20.58%
XLF Financials,-7.73%,-2.77%
XLI Industrials,-13.94%,-11.41%
XLY Cons. Disc.,-2.60%,-1.38%
Pooled,-6.24%,-0.59%


## Diagnostics: largest level errors

Inspect the five largest ETF-month errors and their shares of full-macro level MSE.


In [28]:
macro_errors = d[d.model.eq("en_full_macro")].copy()
macro_errors["level_ratio"] = macro_errors.fc_level / macro_errors.actual_RV
macro_errors["level_error_share"] = macro_errors.MSE_level / macro_errors.MSE_level.sum()
top5 = macro_errors.nlargest(5, "MSE_level")
error_cols = ["etf", "date", "target_date", "actual_RV", "fc_log", "sigma2", "fc_level",
              "level_ratio", "MSE_log", "MSE_level", "QLIKE", "level_error_share"]

top5[error_cols]

,etf,date,target_date,actual_RV,fc_log,sigma2,fc_level,level_ratio,MSE_log,MSE_level,QLIKE,level_error_share
88,XLY,2008-10-31,2008-11-30,0.045574,-0.528812,0.329611,0.694887,15.247411,6.551574,0.421607,1.789995,0.366374
84,XLF,2008-10-31,2008-11-30,0.087358,-0.929827,0.714462,0.564059,6.456855,2.273796,0.227244,1.020017,0.197474
86,XLI,2008-10-31,2008-11-30,0.024007,-1.069012,0.314917,0.401899,16.741049,7.077757,0.142803,1.877597,0.124095
134,XLF,2009-03-31,2009-04-30,0.062968,-1.343476,0.751981,0.380037,6.035425,2.021106,0.100533,0.963335,0.087363
80,XLB,2008-10-31,2008-11-30,0.035719,-1.584606,0.385108,0.248565,6.958862,3.053623,0.045303,1.083718,0.039368
